In [0]:
%restart_python

In [0]:
import numpy as np
import pandas as pd

import pyspark.sql.types as T
import pyspark.sql.functions as F

In [0]:
dbx_params = dbutils.notebook.entry_point.getCurrentBindings()

NSAMPLES = int(dbx_params.get("refresh_raw_data") or 128)
NUM_FEATURES = int(dbx_params.get("num_features") or 64)

In [0]:
input_matrix = np.random.rand(NSAMPLES, NUM_FEATURES)
df = spark.createDataFrame(
    pd.DataFrame(
        {
            **{
                f"feature_{i}": input_matrix[:, i].tolist()
                for i in range(NUM_FEATURES)
            },
            "target": np.random.randint(2, size=NSAMPLES).tolist(),
        }
    )
)
del input_matrix

In [0]:
select_cols = [f"feature_{i}" for i in range(NUM_FEATURES)]
df = df.withColumn(
    "sparse_vector",
    F.create_map(
        *list(
            sum(
                [
                    (F.lit(fid).cast("int"), F.col(f"`{fname}`").cast("double"))
                    for fid, fname in enumerate(select_cols)
                ],
                (),
            )
        )
    )
).drop(*select_cols).withColumnRenamed("sparse_vector", "features")
del select_cols

In [0]:
df.select(
    F.col("features").cast(T.MapType(T.IntegerType(), T.FloatType())).alias("features"),
    F.col("target").cast(T.ShortType()).alias("target"),
).write.mode("overwrite").saveAsTable("dlh.tmp.demo_processed")

In [0]:
spark.table("dlh.tmp.demo_processed").printSchema()